In [1]:
import sys
import time
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path(r"d:\ML Project - E-commerce product search with semantic ranking")
DATA = ROOT / "data"
sys.path.append(str(ROOT))

corpus = pd.read_parquet(DATA / "corpus.parquet")
queries = pd.read_parquet(DATA / "queries.parquet")

print("Corpus:", len(corpus))
print("Queries:", len(queries))

Corpus: 40063
Queries: 500


In [2]:
from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = 128   # shorter text = much faster

print("Device:", device)
print("Embedding size:", model.get_sentence_embedding_dimension())

d:\ML Project - E-commerce product search with semantic ranking\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2103.29it/s]


Device: cpu
Embedding size: 384


C:\Users\vansh\AppData\Local\Temp\ipykernel_8264\59321840.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding size:", model.get_sentence_embedding_dimension())


In [3]:
test_texts = corpus["search_text"].head(5).tolist()

start = time.time()
vecs = model.encode(test_texts, normalize_embeddings=True)
print("Shape:", vecs.shape)
print(f"Time: {time.time() - start:.2f} s")

Shape: (5, 384)
Time: 0.41 s


In [4]:
from sentence_transformers import util

q = model.encode("something to keep my coffee hot", normalize_embeddings=True)
docs = [
    "Insulated stainless steel travel mug, keeps drinks hot for 12 hours",
    "Coffee table book about world cuisine",
    "Hot pink yoga mat",
]
d = model.encode(docs, normalize_embeddings=True)

scores = util.cos_sim(q, d)[0]
for doc, s in zip(docs, scores):
    print(f"{float(s):.3f}  {doc}")

0.546  Insulated stainless steel travel mug, keeps drinks hot for 12 hours
0.389  Coffee table book about world cuisine
0.207  Hot pink yoga mat


In [5]:
start = time.time()

corpus_emb = model.encode(
    corpus["search_text"].tolist(),
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print("Shape:", corpus_emb.shape)
print(f"Done in {(time.time() - start) / 60:.1f} minutes")

Batches: 100%|██████████| 626/626 [19:52<00:00,  1.90s/it]


Shape: (40063, 384)
Done in 19.9 minutes


In [6]:
np.save(DATA / "corpus_embeddings.npy", corpus_emb.astype("float32"))
print("Saved!")

Saved!


In [7]:
query_emb = model.encode(
    queries["query"].tolist(),
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)

np.save(DATA / "query_embeddings.npy", query_emb.astype("float32"))
print("Query embeddings:", query_emb.shape)

Batches: 100%|██████████| 8/8 [00:01<00:00,  4.21it/s]

Query embeddings: (500, 384)


In [8]:
loaded = np.load(DATA / "corpus_embeddings.npy")
print("Embeddings:", loaded.shape[0], "| Corpus rows:", len(corpus))
assert loaded.shape[0] == len(corpus), "Row count mismatch!"
print("Row i of the embeddings = row i of corpus.parquet. Do not re-sort the corpus.")

Embeddings: 40063 | Corpus rows: 40063
Row i of the embeddings = row i of corpus.parquet. Do not re-sort the corpus.
